<a href="https://colab.research.google.com/github/aslestia/Published_article/blob/main/acs_manuscript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
OUTDIR = f"/content/drive/MyDrive/rl_results/{RUN_ID}"
os.makedirs(OUTDIR, exist_ok=True)
print("OUTDIR:", OUTDIR)


Mounted at /content/drive
OUTDIR: /content/drive/MyDrive/rl_results/20260215_014140


#CELL 1 — Import + seed util

In [ ]:
import numpy as np
import pandas as pd
import random
import gc

def set_seed(seed:int):
    random.seed(seed)
    np.random.seed(seed)


#CELL 2 — Ambil data & bentuk returns

In [ ]:
import yfinance as yf

tickers = ["AAPL","AMZN","GOOG","JPM","META","MSFT","NFLX","NVDA","TSLA","V"]
start, end = "2019-01-01", "2024-11-01"

prices = yf.download(tickers, start=start, end=end)["Close"].dropna()
returns = prices.pct_change().dropna()

print("returns shape:", returns.shape)
returns.head()

/tmp/ipython-input-2154197908.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(tickers, start=start, end=end)["Close"].dropna()
[*********************100%***********************]  10 of 10 completed

returns shape: (1468, 10)


Ticker,AAPL,AMZN,GOOG,JPM,META,MSFT,NFLX,NVDA,TSLA,V
Date,,,,,,,,,,
2019-01-03,-0.099608,-0.025241,-0.028484,-0.014212,-0.029039,-0.036788,0.013226,-0.060417,-0.031472,-0.036036
2019-01-04,0.042689,0.050064,0.053786,0.036866,0.047138,0.046509,0.097234,0.064067,0.057697,0.043081
2019-01-07,-0.002226,0.034353,-0.002167,0.000695,0.000725,0.001276,0.059717,0.052941,0.054361,0.018032
2019-01-08,0.019063,0.016612,0.007385,-0.001886,0.032452,0.007251,0.015634,-0.024895,0.001164,0.005439
2019-01-09,0.016982,0.001714,-0.001505,-0.001690,0.011927,0.014299,-0.000968,0.019667,0.009483,0.011769


#CELL 3 — Split train/eval (chronological)

In [ ]:
split_ratio = 0.8
T = len(returns)
cut = int(split_ratio*T)

returns_train = returns.iloc[:cut].copy()
returns_eval  = returns.iloc[cut:].copy()

print("Train length:", len(returns_train), "Eval length:", len(returns_eval))


Train length: 1174 Eval length: 294


#CELL 4 — Environment (PENTING: tanpa registry gym/gymnasium)

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from gymnasium.envs.registration import register
from sklearn.preprocessing import StandardScaler

class MultiStockReturnEnv(gym.Env):
    metadata = {'render.modes': ['human']}

    def __init__(self, returns_df, initial_balance=10000, transaction_cost_pct=0.001,
                 reward_scale=1e-3, max_share_unit=1.0, clip_ret=0.2, debug=False):
        super().__init__()
        # data
        self.raw_df = returns_df.reset_index(drop=True).copy()
        self.df = self.raw_df.fillna(0.0).astype(float)
        self.n_stocks = self.df.shape[1]

        # params
        self.initial_balance = float(initial_balance)
        self.transaction_cost_pct = float(transaction_cost_pct)
        self.reward_scale = float(reward_scale)
        self.max_share_unit = float(max_share_unit)  # scaling factor for holdings
        self.clip_ret = float(clip_ret)
        self.debug = bool(debug)

        # observation: normalized returns (n_stocks) + cash_norm + shares_norm_mean
        obs_dim = self.n_stocks + 2
        high = np.ones(obs_dim, dtype=np.float32) * np.finfo(np.float32).max
        self.observation_space = spaces.Box(low=-high, high=high, dtype=np.float32)

        # discrete action space (easy baseline)
        self.action_space = spaces.Discrete(3)
        self.scaler = StandardScaler()
        self.scaler.fit(self.df.values)

        # state
        self.reset()

    def _get_obs(self):
        row = np.nan_to_num(self.df.loc[self.current_step].values, nan=0.0).astype(np.float32)
        row = np.clip(row, -self.clip_ret, self.clip_ret)

        try:
            row_scaled = self.scaler.transform(row.reshape(1, -1)).flatten()
        except Exception:
            row_scaled = row

        cash_norm = np.clip(self.balance / (self.initial_balance * 10.0), -1e3, 1e3)  # bounded
        shares_norm = np.clip(np.mean(self.shares_held) / (self.max_share_unit + 1e-9), -1e3, 1e3)
        obs = np.concatenate([row_scaled.astype(np.float32), np.array([cash_norm, shares_norm], dtype=np.float32)])
        obs = np.nan_to_num(obs, nan=0.0, posinf=1e9, neginf=-1e9)
        return obs

    def step(self, action):
        info = {}

        rets = np.nan_to_num(self.df.loc[self.current_step].values, nan=0.0).astype(np.float64)
        rets = np.clip(rets, -self.clip_ret, self.clip_ret)

        prev_total = float(self.total_asset)

        if action == 1:  # buy all available cash, proportional across stocks
            cash_available = self.balance * (1.0 - self.transaction_cost_pct)
            units_per_stock = (cash_available / self.initial_balance) / max(1.0, self.n_stocks)
            self.shares_held += units_per_stock
            self.balance = 0.0
        elif action == 2:  # sell all (liquidate)
            mean_rets = np.mean(rets)
            proceeds = np.sum(self.shares_held) * self.initial_balance * (1.0 + mean_rets)
            proceeds *= (1.0 - self.transaction_cost_pct)
            self.balance += proceeds
            self.shares_held[:] = 0.0

        try:
            self.shares_held = self.shares_held * (1.0 + rets)
        except Exception:
            if np.isscalar(self.shares_held):
                self.shares_held = np.array([self.shares_held] * self.n_stocks)
                self.shares_held = self.shares_held * (1.0 + rets)
            else:
                self.shares_held = np.zeros(self.n_stocks)

        self.shares_held = np.nan_to_num(self.shares_held, nan=0.0, posinf=1e9, neginf=-1e9)
        self.total_asset = float(self.balance + np.sum(self.shares_held) * self.initial_balance)

        reward = (self.total_asset - prev_total) * self.reward_scale
        reward = float(np.nan_to_num(reward, nan=0.0))
        reward = float(np.clip(reward, -1.0, 1.0))

        if self.debug:
            if (np.isnan(reward) or np.isnan(self.total_asset) or np.isnan(self.shares_held).any()):
                print(f"[DEBUG NaN] step={self.current_step} reward={reward} total={self.total_asset}")
            if abs(reward) > 0.5:
                print(f"[DEBUG LargeReward] step={self.current_step} reward={reward} total={self.total_asset}")

        self.current_step += 1
        done = False
        if self.current_step >= len(self.df) - 1:
            done = True

        obs = self._get_obs()
        terminated, truncated = done, False
        return obs, reward, terminated, truncated, info

    def reset(self, seed=None, options=None):
        try:
            super().reset(seed=seed)
        except TypeError:
            pass
        self.current_step = 0
        self.balance = float(self.initial_balance)
        self.shares_held = np.zeros(self.n_stocks, dtype=np.float64)
        self.total_asset = float(self.initial_balance)
        obs = self._get_obs()
        return obs, {}

    def render(self, mode='human'):
        print(f"Step {self.current_step} | Total Asset: {self.total_asset:.2f}")

tickers = ['AAPL', 'MSFT', 'GOOG',
           'AMZN', 'META', 'TSLA',
           'NVDA', 'JPM', 'V', 'NFLX']

dfs = []
for t in tickers:
    df_t = yf.download(t, period='5y', interval='1d', progress=False)
    df_t = df_t[['Close']].rename(columns={'Close': t})
    dfs.append(df_t)

df_all = pd.concat(dfs, axis=1).dropna()
returns = df_all.pct_change().dropna().reset_index(drop=True)

env_id = "MultiStockReturn-v0"
if env_id in gym.registry:
    del gym.registry[env_id]

register(
    id=env_id,
    entry_point=lambda:
        MultiStockReturnEnv(
            returns_df=returns,
            reward_scale=1e-3
        ),
    max_episode_steps=len(returns) - 1,
)

print("Registered envs:", [spec.id for spec in gym.registry.values() if "MultiStockReturn" in spec.id])

env = gym.make(env_id)
obs, _ = env.reset()
print("obs shape:", obs.shape, "obs sample:", obs[:5])
step_out = env.step(0)
print("step output lengths:", len(step_out))

/tmp/ipython-input-13389218.py:125: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_t = yf.download(t, period='5y', interval='1d', progress=False)
/tmp/ipython-input-13389218.py:125: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_t = yf.download(t, period='5y', interval='1d', progress=False)
/tmp/ipython-input-13389218.py:125: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_t = yf.download(t, period='5y', interval='1d', progress=False)
/tmp/ipython-input-13389218.py:125: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_t = yf.download(t, period='5y', interval='1d', progress=False)
/tmp/ipython-input-13389218.py:125: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_t = yf.download(t, period='5y', interval='1d', progress=False)
/tmp/ipython-input-13389218.py:125: FutureWarning: YF.download() has changed argument auto

Registered envs: ['MultiStockReturn-v0']
obs shape: (12,) obs sample: [-1.0488174   0.2298209   0.10284568  0.5287684  -0.09179906]
step output lengths: 5


#CELL 5 — Training functions (copy paste dari notebook lama)

##RunningNorm

In [ ]:
class RunningNorm:
    def __init__(self, eps=1e-5):
        self.mean = 0.0
        self.var = 1.0
        self.count = eps

    def update(self, x):
        x = np.array(x)
        batch_mean = x.mean()
        batch_var = x.var()
        batch_count = 1

        delta = batch_mean - self.mean
        tot_count = self.count + batch_count

        new_mean = self.mean + delta * batch_count / tot_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + delta**2 * self.count * batch_count / tot_count
        new_var = M2 / tot_count

        self.mean = new_mean
        self.var = new_var
        self.count = tot_count

    def norm(self, x):
        return (x - self.mean) / (np.sqrt(self.var) + 1e-8)


##TAMBAHAN

In [ ]:
import numpy as np
import torch
import torch.nn as nn

def reset_env(env, seed=None):
    # gymnasium reset() -> (obs, info)
    out = env.reset(seed=seed)
    if isinstance(out, tuple) and len(out) == 2:
        return out[0]
    return out

def step_env(env, action):
    # gymnasium step() -> (obs, reward, terminated, truncated, info)
    out = env.step(action)
    if len(out) == 5:
        obs, r, terminated, truncated, info = out
        done = terminated or truncated
        return obs, r, done, truncated, info
    # fallback (gym lama)
    obs, r, done, info = out
    return obs, r, done, False, info

def linear_schedule(start, end, t, T):
    if T <= 0:
        return float(end)
    frac = min(max(t / T, 0.0), 1.0)
    return float(start + frac * (end - start))

class Meter:
    def __init__(self, window=20):
        self.window = int(window)
        self.rewards = []

    def push(self, r):
        self.rewards.append(float(r))

    def get_last_moving_avg(self):
        if len(self.rewards) == 0:
            return 0.0
        w = min(self.window, len(self.rewards))
        return float(np.mean(self.rewards[-w:]))

    def moving_avg(self):
        # optional helper
        out = []
        for i in range(len(self.rewards)):
            w = min(self.window, i+1)
            out.append(float(np.mean(self.rewards[i+1-w:i+1])))
        return out

def make_optimizer(params, opt_name='adam', lr=1e-3, weight_decay=0.0):
    opt_name = opt_name.lower()
    if opt_name == 'adam':
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if opt_name == 'adamw':
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if opt_name == 'rmsprop':
        return torch.optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    if opt_name == 'sgd':
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {opt_name}")


##REINFORCE

In [ ]:
import torch
import torch.nn as nn
import math

class PolicyNet(nn.Module):
    """Jaringan Kebijakan murni (untuk REINFORCE tanpa baseline V-value)."""
    def __init__(self, obs_dim, act_dim, hidden=128, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.LayerNorm(hidden),
            nn.LeakyReLU(0.1),

            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.LeakyReLU(0.1),

            nn.Dropout(dropout),
            nn.Linear(hidden, act_dim)
        )

        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=1.0)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        return torch.distributions.Categorical(logits=self.net(x))

class PolicyBaseline(nn.Module):
    """Jaringan Policy dan Value (untuk REINFORCE dengan baseline V-value)."""
    def __init__(self, obs_dim, act_dim, hidden=128):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU())
        self.pi_head = nn.Linear(hidden, act_dim)
        self.v_head  = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.body(x)
        dist = torch.distributions.Categorical(logits=self.pi_head(h))
        v    = self.v_head(h).squeeze(-1)
        return dist, v

# --- Core RL Calculations ---

def compute_returns(rewards, gamma):
    """Menghitung discounted returns (G_t)."""
    G, out = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        out.append(G)
    out.reverse()
    return torch.tensor(out, dtype=torch.float32)

def compute_cvar_value(returns, alpha=0.1):
    """Menghitung CVaR dari returns (bukan loss)."""
    sorted_returns, _ = torch.sort(returns)
    n = len(sorted_returns)
    k = max(1, math.ceil(alpha * n))
    cvar = sorted_returns[:k].mean()
    return cvar


# --- Training Function ---
def train_reinforce(env, episodes=800, gamma=0.99, seed=42,
                                   opt_name='adam', lr=1e-3, weight_decay=0.0, hidden=256,
                                   use_baseline=True,
                                   var=False, cvar=False, evar=False,
                                   alpha=0.1, beta_risk=0.5, # alpha untuk VaR/CVaR, beta_risk untuk EVaR
                                   entropy_start=0.02, entropy_end=0.001,
                                   grad_clip=0.5, render=False, window=20,
                                   csv_filename='reinforce_risk_log.csv'):

    # -----------------------------------------------------------
    # PENGECEKAN KENDALA RISIKO (CONSTRAINTS)
    # -----------------------------------------------------------
    risk_options = [var, cvar, evar]
    active_risks = sum(risk_options)

    if active_risks > 1:
        print("-------------------------------------------------------------------------------------------")
        print("ERROR: Hanya boleh memilih SATU ukuran risiko (VAR, CVAR, atau EVAR) aktif pada satu waktu.")
        print("-------------------------------------------------------------------------------------------")
        return [], [], None

    if var: risk_measure = 'var'
    elif cvar: risk_measure = 'cvar'
    elif evar: risk_measure = 'evar'
    else: risk_measure = None

    if use_baseline:
        tag = 'REINFORCE-BL'
        if risk_measure: tag += f"-{risk_measure.upper()}"
    else:
        tag = 'REINFORCE'
        if risk_measure: tag += f"-{risk_measure.upper()}"

    # The previous diff introduced an indentation error and used an undefined env_name. Reverting to safe check.
    env_name_str = env.spec.id if hasattr(env, 'spec') and env.spec else "CustomEnv" # Safely get env name
    print(f"Memulai pelatihan {tag} di {env_name_str}.")


    # --- Setup Awal ---
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Menggunakan perangkat: {device}")

    #env = gym.make(env_name)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n
    stnorm = RunningNorm()

    if use_baseline:
        net = PolicyBaseline(obs_dim, act_dim, hidden).to(device)
    else:
        net = PolicyNet(obs_dim, act_dim, hidden).to(device)

    params = net.parameters()
    opt = make_optimizer(params, opt_name=opt_name, lr=lr, weight_decay=weight_decay)
    meter = Meter(window=window)

    log_data = []

    # --- Training Loop ---
    for ep in range(episodes):
        obs = reset_env(env, seed+ep)
        done, total_r = False, 0.0

        logps, rewards, entros, values = [], [], [], []

        while not done:
            stnorm.update(obs)
            obs_t = torch.tensor(stnorm.norm(obs), dtype=torch.float32, device=device).unsqueeze(0)

            if use_baseline:
                dist, v = net(obs_t)
                values.append(v.squeeze(0))
            else:
                dist = net(obs_t)

            a = dist.sample()

            logps.append(dist.log_prob(a).squeeze(0))
            entros.append(dist.entropy().squeeze(0))

            nobs, r, done, truncated, info = step_env(env, a.item())
            done = done or truncated
            rewards.append(float(r))
            total_r += r
            obs = nobs

        # --- Perhitungan Loss ---

        # 1. Hitung Discounted Returns (G)
        returns = compute_returns(rewards, gamma).to(device)

        # 2. Hitung Advantage (Advantage) atau Normalized Return
        if use_baseline:
            values_t = torch.stack(values)
            adv = returns - values_t.detach() # Baseline G - V
        else:
            # Default: Risk Neutral REINFORCE
            adv = (returns - returns.mean()) / (returns.std() + 1e-8)

        # 3. PENYESUAIAN RISIKO (RISK-SENSITIVE POLICY GRADIENT)
        if risk_measure:

            # Tentukan Baseline Risiko:
            if risk_measure == 'cvar':
                risk_baseline = compute_cvar_value(returns, alpha)
            elif risk_measure == 'var':
                risk_baseline = torch.quantile(returns, alpha)
            elif risk_measure == 'evar':
                # Untuk EVaR, kita menggunakan mean return sebagai baseline untuk perhitungan adv
                # tapi penyesuaian utama ada di weighting (Langkah 3b)
                risk_baseline = returns.mean()

            # 3a. Re-baseline Advantage menggunakan Risk Baseline (Jika tanpa V-value)
            if not use_baseline and risk_measure in ['cvar', 'var', 'evar']:
                # Hentikan normalisasi Mean/Std standar, dan gunakan Risk Baseline
                adv = returns - risk_baseline
                # Normalisasi adv hasil risk-baseline untuk stabilitas (penting!)
                adv = (adv - adv.mean()) / (adv.std() + 1e-8)


            # 3b. Weighting / Masking
            if risk_measure == 'cvar':
                # CVaR: HANYA fokus pada pembaruan dari returns yang berada di kuantil terburuk (alpha) (bagian kiri, kuantil bawah)
                sorted_returns, _ = torch.sort(returns, descending=False)
                k = max(1, math.ceil(alpha * len(returns)))
                cvar_threshold = sorted_returns[k-1]

                # Masking: Advantage hanya aktif (1.0) untuk langkah yang Returns-nya di bawah threshold
                mask = returns <= cvar_threshold
                adv = adv * mask.float()

            elif risk_measure == 'var':
                # VaR: HANYA fokus pada pembaruan dari returns yang lebih buruk dari VaR (kuantil alpha)
                var_threshold = torch.quantile(returns, alpha)

                # Masking: Advantage hanya aktif (1.0) untuk langkah yang Returns-nya di bawah VaR
                mask = returns <= var_threshold
                adv = adv * mask.float()

            elif risk_measure == 'evar':
                adv = adv * torch.exp(beta_risk * adv.detach()) # Exponential weighting


        # Hitung Policy Loss
        policy_loss = -(torch.stack(logps) * adv.detach()).sum()

        # Hitung Value Loss (Hanya jika menggunakan PolicyBaseline)
        if use_baseline:
            value_loss = 0.5 * (adv.pow(2)).sum()
            loss_core = policy_loss + value_loss
        else:
            value_loss = torch.tensor(0.0, device=device)
            loss_core = policy_loss

        # Hitung Entropy Loss
        ent = torch.stack(entros).sum()
        warmup = int(0.7 * episodes)
        if ep <= warmup:
            beta = linear_schedule(entropy_start, entropy_end, ep, warmup)
        else:
            beta = entropy_end

        loss = loss_core - beta * ent

        # Backpropagation
        opt.zero_grad()
        loss.backward()
        if grad_clip is not None:
            nn.utils.clip_grad_norm_(params, grad_clip)
        opt.step()

        # Logging dan Monitoring
        meter.push(total_r)
        current_avg = meter.get_last_moving_avg()

        # Kumpulkan data log
        log_data.append({
            'episode': ep + 1,
            'reward': total_r,
            'moving_avg': current_avg,
            'policy_loss': policy_loss.item(),
            'value_loss': value_loss.item(),
            'total_loss': loss.item(),
            'entropy_beta': beta,
            'risk_measure': risk_measure if risk_measure else 'None',
        })

        if (ep+1) % 25 == 0:
            risk_info = f"| Risk: {risk_measure.upper()}" if risk_measure else ""
            print(f'[{tag}] Ep {ep+1}/{episodes} | R:{total_r:.1f} | Avg({window}): {current_avg:.1f} | Ent:{beta:.4f} {risk_info}')

            # Make CartPole-v1 check safe for environments without spec
            if hasattr(env, 'spec') and env.spec and env.spec.id == 'CartPole-v1' and len(meter.rewards) >= 100 and np.mean(meter.rewards[-100:]) >= 475:
                print(f"[{tag}] Early stop: moving-avg(100) >= 475")
                break


    env.close()
    if csv_filename and log_data:
        fieldnames = log_data[0].keys()
        try:
            import csv # Import csv here if it's not global
            with open(csv_filename, 'w', newline='') as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(log_data)
            print(f"\n[{tag}] Pelatihan Selesai. Data log disimpan ke: {csv_filename}")
        except Exception as e:
            print(f"\n[{tag}] ERROR saat menyimpan CSV: {e}")

    # Return a DataFrame directly
    return pd.DataFrame(log_data)

##A2C

In [ ]:
import torch
import torch.nn as nn
import math
import pandas as pd # Add pandas import for DataFrame logging
import csv # Add csv import for logging

class ActorCriticNet(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=256):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU())
        self.pi_head = nn.Linear(hidden, act_dim)
        self.v_head  = nn.Linear(hidden, 1)
    def forward(self, x):
        h = self.body(x)
        dist = torch.distributions.Categorical(logits=self.pi_head(h))
        v    = self.v_head(h).squeeze(-1)
        return dist, v

def train_actor_critic_batched(env, episodes=800, gamma=0.99, seed=7,
                               opt_name='adam', lr=1e-3, weight_decay=0.0, hidden=256,
                               value_loss_coeff=0.7, entropy_start=0.02, entropy_end=0.005,
                               grad_clip=0.5, render=False, window=20,
                               var=False, cvar=False, evar=False, # New risk parameters
                               alpha=0.1, beta_risk=0.5,           # New risk parameters
                               csv_filename='a2c_risk_log.csv'): # New CSV logging parameter

    # -----------------------------------------------------------
    # PENGECEKAN KENDALA RISIKO (CONSTRAINTS)
    # -----------------------------------------------------------
    risk_options = [var, cvar, evar]
    active_risks = sum(risk_options)

    if active_risks > 1:
        print("-------------------------------------------------------------------------------------------")
        print("ERROR: Hanya boleh memilih SATU ukuran risiko (VAR, CVAR, atau EVAR) aktif pada satu waktu.")
        print("-------------------------------------------------------------------------------------------")
        # Returning an empty DataFrame to match the expected return type
        return pd.DataFrame()

    if var: risk_measure = 'var'
    elif cvar: risk_measure = 'cvar'
    elif evar: risk_measure = 'evar'
    else: risk_measure = None

    tag = 'A2C'
    if risk_measure: tag += f"-{risk_measure.upper()}"

    env_name_str = env.spec.id if hasattr(env, 'spec') and env.spec else "CustomEnv"
    print(f"Memulai pelatihan {tag} di {env_name_str}.")


    set_seed(seed)
    device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Menggunakan perangkat: {device}")

    obs_dim  = env.observation_space.shape[0]
    act_dim  = env.action_space.n
    stnorm   = RunningNorm()

    net   = ActorCriticNet(obs_dim, act_dim, hidden).to(device)
    opt   = make_optimizer(net.parameters(), opt_name=opt_name, lr=lr, weight_decay=weight_decay)
    meter = Meter(window=window)

    log_data = [] # Initialize log_data

    for ep in range(episodes):
        obs = reset_env(env, seed+ep)
        done, total_r = False, 0.0
        logps, values, rewards, entros = [], [], [], []

        while not done:
            stnorm.update(obs)
            obs_t = torch.tensor(stnorm.norm(obs), dtype=torch.float32, device=device).unsqueeze(0)
            dist, v = net(obs_t)
            a = dist.sample()
            logps.append(dist.log_prob(a).squeeze(0))
            entros.append(dist.entropy().squeeze(0))
            values.append(v.squeeze(0))

            nobs, r, done_step, truncated, info = step_env(env, a.item()) # Get all 5 step outputs
            done = done_step or truncated # Update done based on both terminated and truncated
            rewards.append(float(r))
            total_r += r

            obs = nobs # Move observation forward

        # --- AFTER EPISODE ENDS ---
        # 1. Compute discounted returns for the episode (G_t)
        returns = compute_returns(rewards, gamma).to(device)

        # 2. Calculate targets for the value network: G_t
        targets = returns

        # 3. Calculate advantages (deltas): G_t - V(s_t)
        values_t = torch.stack(values)
        deltas = targets - values_t # This is our advantage, A_t

        # 4. PENYESUAIAN RISIKO (RISK-SENSITIVE POLICY GRADIENT)
        if risk_measure:
            # 4b. Weighting / Masking
            if risk_measure == 'cvar':
                # CVaR: HANYA fokus pada pembaruan dari returns yang berada di kuantil terburuk (alpha)
                sorted_returns, _ = torch.sort(returns, descending=False)
                k = max(1, math.ceil(alpha * len(returns)))
                cvar_threshold = sorted_returns[k-1]

                # Masking: Advantage only active for steps where Returns are below the threshold
                mask = returns <= cvar_threshold
                deltas = deltas * mask.float()

            elif risk_measure == 'var':
                # VaR: HANYA fokus pada pembaruan dari returns yang lebih buruk dari VaR (kuantil alpha)
                var_threshold = torch.quantile(returns, alpha)

                # Masking: Advantage only active for steps where Returns are below VaR
                mask = returns <= var_threshold
                deltas = deltas * mask.float()

            elif risk_measure == 'evar':
                # For EVaR, we apply exponential weighting to the advantages.
                deltas = deltas * torch.exp(beta_risk * deltas.detach())

        beta = linear_schedule(entropy_start, entropy_end, ep, episodes-1)
        actor_loss  = -(torch.stack(logps) * deltas.detach()).sum() # deltas already has risk adjustment
        critic_loss = (deltas.pow(2)).sum() * value_loss_coeff # Value loss is still based on squared advantage
        entropy_loss = -beta * torch.stack(entros).sum()
        loss = actor_loss + critic_loss + entropy_loss

        opt.zero_grad(); loss.backward()
        if grad_clip is not None:
            nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
        opt.step()

        meter.push(total_r)
        current_avg = meter.get_last_moving_avg()

        # Kumpulkan data log
        log_data.append({
            'episode': ep + 1,
            'reward': total_r,
            'moving_avg': current_avg,
            'actor_loss': actor_loss.item(),
            'critic_loss': critic_loss.item(),
            'total_loss': loss.item(),
            'entropy_beta': beta,
            'risk_measure': risk_measure if risk_measure else 'None',
        })

        if (ep+1) % 50 == 0:
            risk_info = f"| Risk: {risk_measure.upper()}" if risk_measure else ""
            print(f'[{tag}] Ep {ep+1}/{episodes} | R:{total_r:.1f} | Avg({meter.window}): {current_avg:.1f} | Ent:{beta:.4f} {risk_info}')

    env.close()

    if csv_filename and log_data:
        fieldnames = log_data[0].keys()
        try:
            with open(csv_filename, 'w', newline='') as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(log_data)
            print(f"\n[{tag}] Pelatihan Selesai. Data log disimpan ke: {csv_filename}")
        except Exception as e:
            print(f"\n[{tag}] ERROR saat menyimpan CSV: {e}")

    # Return a DataFrame directly
    return pd.DataFrame(log_data)

#CELL 6 — Runner multi-seed (langsung simpan per seed ke Drive)

In [ ]:
def run_one(seed, algo, obj, episodes=800):
    set_seed(seed)

    # env Anda TIDAK punya argumen risk_measure → jangan kirim itu
    env_tr = MultiStockReturnEnv(returns_train, reward_scale=1e-3)

    # mapping objective → flags (karena risk handling ada di training function)
    var_flag  = (obj == "var")
    cvar_flag = (obj == "cvar")
    evar_flag = (obj == "evar")
    # For 'rn' (risk-neutral), all risk flags should be False
    if obj == "rn":
        var_flag = False
        cvar_flag = False
        evar_flag = False

    # Default alpha and beta_risk values (can be made configurable if needed)
    alpha_val = 0.1
    beta_risk_val = 0.5

    if algo == "reinforce":
        # sesuaikan nama argumen csv kalau di fungsi Anda: csv_filename atau csv_file
        csv_path = os.path.join(OUTDIR, f"log_reinforce_{obj}_seed{seed}.csv")

        df_log = train_reinforce(
            env=env_tr,                 # ❤
 pastikan train_reinforce Anda sudah menerima env (bukan env_name)
            episodes=episodes,
            seed=seed,
            use_baseline=True,
            var=var_flag, cvar=cvar_flag, evar=evar_flag,
            alpha=alpha_val, # Pass alpha
            beta_risk=beta_risk_val, # Pass beta_risk
            csv_filename=csv_path
        )

    elif algo == "a2c":
        csv_path = os.path.join(OUTDIR, f"log_a2c_{obj}_seed{seed}.csv")

        df_log = train_actor_critic_batched( # This function needs to be updated in its cell
            env=env_tr,
            episodes=episodes,
            seed=seed,
            var=var_flag, cvar=cvar_flag, evar=evar_flag,
            alpha=alpha_val, # Pass alpha
            beta_risk=beta_risk_val, # Pass beta_risk
            csv_filename=csv_path
        )
    else:
        raise ValueError("algo harus 'reinforce' atau 'a2c'")

    # ringkasan (hemat RAM)
    # Ensure df_log is a DataFrame before trying to access columns
    if not df_log.empty:
        last20 = float(pd.Series(df_log["reward"]).tail(20).mean())
    else:
        last20 = 0.0 # Or np.nan, depending on desired behavior for failed runs


    return {
        "seed": seed,
        "algo": algo,
        "obj": obj,
        "final_reward_last20": last20,
        "csv_log": csv_path
    }

#CELL 7 — Jalankan multi-seed minimal (reviewer-proof, hemat waktu)

In [ ]:
SEEDS = [0,1,2,3,4]
ALGOS = ["REINFORCE","a2c"]
OBJS  = ["rn", "cvar"]   # bisa ganti "evar" kalau Anda mau

EPISODES = 800

rows = []
for algo in ALGOS:
    for obj in OBJS:
        for seed in SEEDS:
            print(f"Running algo={algo} obj={obj} seed={seed}")
            out = run_one(seed, algo, obj, episodes=EPISODES)
            rows.append(out)

            # simpan summary setiap seed juga
            pd.DataFrame(rows).to_csv(os.path.join(OUTDIR, "multiseed_summary_partial.csv"), index=False)

            gc.collect()

df_sum = pd.DataFrame(rows)
df_sum.to_csv(os.path.join(OUTDIR, "multiseed_summary.csv"), index=False)
df_sum


Running algo=a2c obj=rn seed=0
Memulai pelatihan A2C di CustomEnv.
Menggunakan perangkat: cpu
[A2C] Ep 50/800 | R:37.2 | Avg(20): 36.4 | Ent:0.0191 
[A2C] Ep 100/800 | R:43.4 | Avg(20): 41.7 | Ent:0.0181 
[A2C] Ep 150/800 | R:61.0 | Avg(20): 65.3 | Ent:0.0172 
[A2C] Ep 200/800 | R:89.3 | Avg(20): 90.9 | Ent:0.0163 
[A2C] Ep 250/800 | R:119.6 | Avg(20): 99.7 | Ent:0.0153 
[A2C] Ep 300/800 | R:128.9 | Avg(20): 134.1 | Ent:0.0144 
[A2C] Ep 350/800 | R:155.3 | Avg(20): 146.9 | Ent:0.0134 
[A2C] Ep 400/800 | R:155.2 | Avg(20): 166.0 | Ent:0.0125 
[A2C] Ep 450/800 | R:171.2 | Avg(20): 171.1 | Ent:0.0116 
[A2C] Ep 500/800 | R:203.0 | Avg(20): 196.4 | Ent:0.0106 
[A2C] Ep 550/800 | R:185.9 | Avg(20): 180.3 | Ent:0.0097 
[A2C] Ep 600/800 | R:198.6 | Avg(20): 201.3 | Ent:0.0088 
[A2C] Ep 650/800 | R:205.5 | Avg(20): 210.2 | Ent:0.0078 
[A2C] Ep 700/800 | R:200.2 | Avg(20): 203.2 | Ent:0.0069 
[A2C] Ep 750/800 | R:214.5 | Avg(20): 212.5 | Ent:0.0059 
[A2C] Ep 800/800 | R:218.4 | Avg(20): 211.2 | 

,seed,algo,obj,final_reward_last20,csv_log
0,0,a2c,rn,211.172426,/content/drive/MyDrive/rl_results/20260215_014...
1,0,a2c,cvar,25.168080,/content/drive/MyDrive/rl_results/20260215_014...


In [ ]:
import os, glob, re
import pandas as pd

OUTDIR = "/content/drive/MyDrive/rl_results/20260214_120457"  # <-- ganti

pattern = os.path.join(OUTDIR, "log_reinforce_*_seed*.csv")
files = sorted(glob.glob(pattern))
print("Found log files:", len(files))
assert len(files) > 0, "Tidak menemukan log_reinforce_*.csv di OUTDIR. Cek path foldernya."

rows = []
for fp in files:
    fn = os.path.basename(fp)
    # contoh nama: log_reinforce_rn_seed0.csv atau log_reinforce_cvar_seed3.csv
    m = re.match(r"log_reinforce_(.+)_seed(\d+)\.csv", fn)
    if not m:
        continue
    obj = m.group(1)
    seed = int(m.group(2))

    df = pd.read_csv(fp)
    if "reward" not in df.columns:
        raise ValueError(f"Kolom 'reward' tidak ada di {fn}. Kolom yang ada: {df.columns}")

    last20 = float(df["reward"].tail(20).mean())
    rows.append({
        "seed": seed,
        "algo": "reinforce",
        "obj": obj,
        "final_reward_last20": last20,
        "csv_log": fp
    })

df_sum = pd.DataFrame(rows).sort_values(["obj","seed"]).reset_index(drop=True)
save_path = os.path.join(OUTDIR, "multiseed_summary.csv")
df_sum.to_csv(save_path, index=False)

print("Saved:", save_path)
print(df_sum)


Found log files: 10
Saved: /content/drive/MyDrive/rl_results/20260214_120457/multiseed_summary.csv
   seed       algo   obj  final_reward_last20  \
0     0  reinforce  cvar             2.154982   
1     1  reinforce  cvar             9.150609   
2     2  reinforce  cvar             7.264959   
3     3  reinforce  cvar             3.024703   
4     4  reinforce  cvar             5.618413   
5     0  reinforce    rn           213.411709   
6     1  reinforce    rn           225.827422   
7     2  reinforce    rn           202.621849   
8     3  reinforce    rn           202.097139   
9     4  reinforce    rn           228.213492   

                                             csv_log  
0  /content/drive/MyDrive/rl_results/20260214_120...  
1  /content/drive/MyDrive/rl_results/20260214_120...  
2  /content/drive/MyDrive/rl_results/20260214_120...  
3  /content/drive/MyDrive/rl_results/20260214_120...  
4  /content/drive/MyDrive/rl_results/20260214_120...  
5  /content/drive/MyDrive/rl_re

In [ ]:
import os, glob, re
import pandas as pd

OUTDIR = "/content/drive/MyDrive/rl_results/20260215_014140"  # <-- ganti ke folder A2C Anda

pattern = os.path.join(OUTDIR, "log_a2c_*_seed*.csv")
files = sorted(glob.glob(pattern))
print("Found log files:", len(files))
assert len(files) > 0, f"Tidak menemukan log_a2c_*_seed*.csv di {OUTDIR}. Cek path foldernya."

rows = []
for fp in files:
    fn = os.path.basename(fp)
    # contoh nama: log_a2c_rn_seed0.csv atau log_a2c_cvar_seed3.csv
    m = re.match(r"log_a2c_(.+)_seed(\d+)\.csv", fn)
    if not m:
        print("Skip (nama tidak match):", fn)
        continue

    obj = m.group(1).strip().lower()   # rn / cvar
    seed = int(m.group(2))

    df = pd.read_csv(fp)

    # cari kolom reward yang tersedia
    reward_col = None
    for cand in ["reward", "ep_reward", "episode_reward", "total_reward"]:
        if cand in df.columns:
            reward_col = cand
            break

    if reward_col is None:
        raise ValueError(
            f"Kolom reward tidak ditemukan di {fn}. Kolom yang ada: {list(df.columns)}"
        )

    last20 = float(df[reward_col].tail(20).mean())

    rows.append({
        "seed": seed,
        "algo": "a2c",
        "obj": obj,
        "final_reward_last20": last20,
        "csv_log": fp
    })

df_sum = (pd.DataFrame(rows)
          .sort_values(["obj","seed"])
          .reset_index(drop=True))

save_path = os.path.join(OUTDIR, "multiseed_summary.csv")
df_sum.to_csv(save_path, index=False)

print("Saved:", save_path)
print(df_sum)


Found log files: 10
Saved: /content/drive/MyDrive/rl_results/20260215_014140/multiseed_summary.csv
   seed algo   obj  final_reward_last20  \
0     0  a2c  cvar            25.168080   
1     1  a2c  cvar            36.118606   
2     2  a2c  cvar            28.881264   
3     3  a2c  cvar            28.967197   
4     4  a2c  cvar            62.271489   
5     0  a2c    rn           211.172426   
6     1  a2c    rn           216.550575   
7     2  a2c    rn           215.697333   
8     3  a2c    rn           219.596418   
9     4  a2c    rn           223.147305   

                                             csv_log  
0  /content/drive/MyDrive/rl_results/20260215_014...  
1  /content/drive/MyDrive/rl_results/20260215_014...  
2  /content/drive/MyDrive/rl_results/20260215_014...  
3  /content/drive/MyDrive/rl_results/20260215_014...  
4  /content/drive/MyDrive/rl_results/20260215_014...  
5  /content/drive/MyDrive/rl_results/20260215_014...  
6  /content/drive/MyDrive/rl_results/20260

In [ ]:
import pandas as pd

# ganti path sesuai folder Anda
df1 = pd.read_csv("/content/drive/MyDrive/rl_results/20260214_120457/multiseed_summary.csv")
df2 = pd.read_csv("/content/drive/MyDrive/rl_results/20260215_014140/multiseed_summary.csv")

df_all = pd.concat([df1, df2], ignore_index=True)

df_all.to_csv("/content/drive/MyDrive/rl_results/multiseed_summary_combined.csv", index=False)

df_all


,seed,algo,obj,final_reward_last20,csv_log
0,0,reinforce,cvar,2.154982,/content/drive/MyDrive/rl_results/20260214_120...
1,1,reinforce,cvar,9.150609,/content/drive/MyDrive/rl_results/20260214_120...
2,2,reinforce,cvar,7.264959,/content/drive/MyDrive/rl_results/20260214_120...
3,3,reinforce,cvar,3.024703,/content/drive/MyDrive/rl_results/20260214_120...
4,4,reinforce,cvar,5.618413,/content/drive/MyDrive/rl_results/20260214_120...
5,0,reinforce,rn,213.411709,/content/drive/MyDrive/rl_results/20260214_120...
6,1,reinforce,rn,225.827422,/content/drive/MyDrive/rl_results/20260214_120...
7,2,reinforce,rn,202.621849,/content/drive/MyDrive/rl_results/20260214_120...
8,3,reinforce,rn,202.097139,/content/drive/MyDrive/rl_results/20260214_120...
9,4,reinforce,rn,228.213492,/content/drive/MyDrive/rl_results/20260214_120...


#CELL 8 — Buat tabel mean ± std + uji signifikansi

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/rl_results/multiseed_summary_combined.csv")

table = (
    df
    .groupby(["algo","obj"])["final_reward_last20"]
    .agg(["mean","std"])
    .reset_index()
)

table


,algo,obj,mean,std
0,a2c,cvar,36.281327,15.060517
1,a2c,rn,217.232811,4.475469
2,reinforce,cvar,5.442733,2.904991
3,reinforce,rn,214.434322,12.373910


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/rl_results/multiseed_summary_combined.csv")

(df.groupby(["algo","obj"])["final_reward_last20"]
   .agg(["count","mean","std"])
   .reset_index())


,algo,obj,count,mean,std
0,a2c,cvar,5,36.281327,15.060517
1,a2c,rn,5,217.232811,4.475469
2,reinforce,cvar,5,5.442733,2.904991
3,reinforce,rn,5,214.434322,12.373910


In [ ]:
from scipy.stats import ttest_rel, wilcoxon

for obj in ["rn","cvar"]:
    a2c = df[(df.algo=="a2c") & (df.obj==obj)].sort_values("seed")["final_reward_last20"]
    rf  = df[(df.algo=="reinforce") & (df.obj==obj)].sort_values("seed")["final_reward_last20"]

    print("OBJ =", obj)
    print("paired t-test p =", ttest_rel(a2c, rf).pvalue)
    print("wilcoxon p      =", wilcoxon(a2c, rf).pvalue)
    print()


OBJ = rn
paired t-test p = 0.6233133262562078
wilcoxon p      = 0.8125

OBJ = cvar
paired t-test p = 0.009131717311685904
wilcoxon p      = 0.0625



Uji signifikansi A2C vs REINFORCE untuk tiap objective:

In [ ]:
for obj in OBJS:
    a2c = df_sum[(df_sum.algo=="a2c") & (df_sum.obj==obj)].sort_values("seed")["final_reward_last20"].values
    rf  = df_sum[(df_sum.algo=="reinforce") & (df_sum.obj==obj)].sort_values("seed")["final_reward_last20"].values

    p_t = ttest_rel(a2c, rf).pvalue
    p_w = wilcoxon(a2c, rf).pvalue

    print(f"OBJ={obj} | paired t-test p={p_t:.4g} | wilcoxon p={p_w:.4g}")


ValueError: Array shapes are incompatible for broadcasting.